# Tech Challenge Fase 4

## Notebook 01: Coleta de Dados

**Objetivo:** Baixar o histórico de preços da ação **PETR4.SA** (Petrobras) usando a biblioteca `yfinance` e salvar como tabela Delta em `jessyca_catalog.postech_fase4`.

### 1. Instalar a biblioteca yfinance

In [0]:
%pip install --upgrade yfinance curl_cffi --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### 2. Parâmetros da coleta

Definir widgets para preenchimento e execução facilitados.

In [0]:
dbutils.widgets.text("ticker", "PETR4.SA", "Ticker (yfinance)")
dbutils.widgets.text("start_date", "2018-01-01", "Data inicial")
dbutils.widgets.text("end_date", "2025-12-31", "Data final")
dbutils.widgets.text("catalog", "jessyca_catalog", "Catalog")
dbutils.widgets.text("schema", "postech_fase4", "Schema")
dbutils.widgets.text("table", "precos_acoes", "Tabela")

TICKER     = dbutils.widgets.get("ticker")
START_DATE = dbutils.widgets.get("start_date")
END_DATE   = dbutils.widgets.get("end_date")
CATALOG    = dbutils.widgets.get("catalog")
SCHEMA     = dbutils.widgets.get("schema")
TABLE      = dbutils.widgets.get("table")

FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"
print(f"Coletando {TICKER} de {START_DATE} até {END_DATE}")
print(f"Destino: {FULL_TABLE}")

Coletando PETR4.SA de 2018-01-01 até 2025-12-31
Destino: jessyca_catalog.postech_fase4.precos_acoes


### 3. Criar Schema e Volume para Armazenar Artefatos

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.artefatos")

Schema e volume prontos.


## 4. Download dos dados via yfinance

Busca dos dados do Yahoo Finance com uso do `curl_cffi` para evitar bloqueio de acesso via `yf.download` e criação de `DataFrame` pandas.

In [0]:
import time
import yfinance as yf
import pandas as pd
from curl_cffi import requests as cffi_requests

def baixar_com_retry(ticker: str, start: str, end: str, tentativas: int = 4) -> pd.DataFrame:
    """Baixa histórico com retry exponencial e session simulando Chrome."""
    last_err = None
    for n in range(1, tentativas + 1):
        try:
            session = cffi_requests.Session(impersonate="chrome")
            tk = yf.Ticker(ticker, session=session)
            df = tk.history(
                start=start, end=end, interval="1d",
                auto_adjust=False, actions=False, raise_errors=True,
            )
            if len(df) > 0:
                return df
            print(f"Tentativa {n}: 0 linhas, repetindo...")
        except Exception as e:
            last_err = e
            print(f"Tentativa {n} falhou: {e}")
        time.sleep(2 ** n)  # 2, 4, 8, 16 s
    raise RuntimeError(f"Falha após {tentativas} tentativas. Último erro: {last_err}")

df_pd = baixar_com_retry(TICKER, START_DATE, END_DATE)

# Normaliza nomes/colunas
df_pd = df_pd.reset_index()
df_pd.columns = [c.lower().replace(" ", "_") for c in df_pd.columns]
df_pd["ticker"] = TICKER
# datetime com timezone → date naive (Spark/Delta lida melhor)
df_pd["date"] = pd.to_datetime(df_pd["date"]).dt.tz_localize(None)

print(f"Registros baixados: {len(df_pd)}")
df_pd.head()

/home/spark-bca6af42-8e1b-4626-a835-ca/.ipykernel/5001/command-7280556912317248-3997179492:13: DeprecationWarning: 'raise_errors' deprecated, do: yf.config.debug.hide_exceptions = False
  df = tk.history(


Registros baixados: 1988


/local_disk0/.ephemeral_nfs/envs/pythonEnv-bca6af42-8e1b-4626-a835-ca092d8f85ab/lib/python3.11/site-packages/yfinance/scrapers/history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


,date,open,high,low,close,adj_close,volume,ticker
0,2018-01-02,16.190001,16.549999,16.190001,16.549999,4.349907,33461800,PETR4.SA
1,2018-01-03,16.490000,16.719999,16.370001,16.700001,4.389334,55940900,PETR4.SA
2,2018-01-04,16.780001,16.959999,16.620001,16.730000,4.397219,37064900,PETR4.SA
3,2018-01-05,16.700001,16.860001,16.570000,16.830000,4.423502,26958200,PETR4.SA
4,2018-01-08,16.740000,17.030001,16.709999,17.030001,4.476068,28400000,PETR4.SA


## 5. Salvar como tabela Delta

Conversão de pandas DataFrame para Spark DataFrame e gravação de tabela.

In [0]:
df_spark = spark.createDataFrame(df_pd)

(
    df_spark.write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(FULL_TABLE)
)

print(f"Tabela salva: {FULL_TABLE}")

Tabela salva: jessyca_catalog.postech_fase4.precos_acoes


## 6. Validação de Primeiras Linhas

In [0]:
display(spark.table(FULL_TABLE).orderBy("date").limit(10))

date,open,high,low,close,adj_close,volume,ticker
2018-01-02T00:00:00.000Z,16.190000534057617,16.549999237060547,16.190000534057617,16.549999237060547,4.349907398223877,33461800,PETR4.SA
2018-01-03T00:00:00.000Z,16.489999771118164,16.719999313354492,16.3700008392334,16.700000762939453,4.389333724975586,55940900,PETR4.SA
2018-01-04T00:00:00.000Z,16.780000686645508,16.959999084472656,16.6200008392334,16.729999542236328,4.397218704223633,37064900,PETR4.SA
2018-01-05T00:00:00.000Z,16.700000762939453,16.860000610351562,16.56999969482422,16.829999923706055,4.423501968383789,26958200,PETR4.SA
2018-01-08T00:00:00.000Z,16.739999771118164,17.030000686645508,16.709999084472656,17.030000686645508,4.476068496704102,28400000,PETR4.SA
2018-01-09T00:00:00.000Z,17.030000686645508,17.15999984741211,16.959999084472656,17.030000686645508,4.476068496704102,35070900,PETR4.SA
2018-01-10T00:00:00.000Z,16.920000076293945,17.049999237060547,16.770000457763672,16.799999237060547,4.415616035461426,28547700,PETR4.SA
2018-01-11T00:00:00.000Z,16.8799991607666,17.299999237060547,16.84000015258789,17.25,4.5338921546936035,37921500,PETR4.SA
2018-01-12T00:00:00.000Z,17.040000915527344,17.40999984741211,17.020000457763672,17.299999237060547,4.54703426361084,45912100,PETR4.SA
2018-01-15T00:00:00.000Z,17.31999969482422,17.440000534057617,17.149999618530273,17.350000381469727,4.560174942016602,28945400,PETR4.SA


In [0]:
total = spark.table(FULL_TABLE).count()
print(f"Total de linhas na tabela: {total}")

Total de linhas na tabela: 1988
